# Spectral Graffiti: Few-Shot Audio In-Painting via Amortized Inference

## 1. The Problem Statement
The recovery of The New Yardbirds' *Spectral Graffiti* master tapes presents a unique missing-data problem. Due to catastrophic Sticky Shed Syndrome, the magnetic tape has physically degraded, leaving microscopic gaps in the 100ms audio samples. This is not a standard noise-reduction task; the signal is entirely missing in these gaps.

Our objective is to perform **Few-Shot Signal In-Painting**: analyzing the surviving audio fragments (context points) to mathematically infer the underlying spectral signature of the instrument and reconstruct the missing waveforms (target points).

---

## 2. Tackling the Core Constraint: Amortized Inference
Audio is fundamentally non-stationary. A single 100ms clip might contain the slow decay of a bass guitar or the high-frequency chaos of a cymbal crash.

To solve this, we built a unified deep learning model that shifts the heavy computational cost entirely to the training phase. By learning the universal physical properties of the audio dataset across 80,000 samples, our model can process an unseen, fragmented audio clip and predict all missing gaps in a **single forward pass**—without any per-sample optimization loops during inference.

---

## 3. The Architecture: Masked  Transformer
To capture the complex temporal dependencies of audio waves, we designed a **Masked Transformer Encoder**.

Our architecture processes the data through the following pipeline:
* **Dual-Feature Input:** Instead of feeding raw voltage, the model receives a tensor with 2 features for each sample. Feature 1 is the masked voltage (true signal where available, `0.0` where missing). Feature 2 is the `Is_Context` binary mask. This explicitly tells the network which points to trust and which to reconstruct.
* **Positional Encoding:** Because audio frequencies are strictly dependent on time, we inject learnable positional embeddings. This ensures the Transformer understands the absolute sequence of the 100ms window.
* **Bidirectional Self-Attention:** Unlike autoregressive language models that only look into the past, our Transformer uses full bidirectional attention. A missing tape gap at $t=50$ dynamically calculates attention weights by looking at surviving context points both before ($t=10$) and after ($t=90$) the gap simultaneously.

---

## 4. The Engine of Learning: Custom Masked MSE Loss
Using standard Mean Squared Error (MSE) over the entire 100ms sequence causes catastrophic failure, as it forces the model to waste learning capacity auto-encoding the context points it was already given.

To force the model to focus strictly on learning the physics of the *missing* audio, we implemented a custom `masked_mse_loss`:

$$\mathcal{L}_{masked}=\frac{\sum_{i=1}^{N}(\hat{y}_i-y_i)^2\cdot(1-m_i)}{\sum_{i=1}^{N}(1-m_i)+\epsilon}$$

By mathematically inverting the `Is_Context` mask $(1 - m_i)$, we zero out the gradients for the known context points. The network's weights update exclusively based on how accurately it predicts the hidden gaps.

---


In [ ]:
import pandas as pd
import torch
import torch.optim as optim


In [ ]:

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from google.colab import files
import io
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))


Saving test_features_spectral - test_features_spectral.csv to test_features_spectral - test_features_spectral.csv


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   Sample_ID   1000000 non-null  int64  
 1   Time_ms     1000000 non-null  int64  
 2   Is_Context  1000000 non-null  int64  
 3   Value       1000000 non-null  float64
dtypes: float64(1), int64(3)
memory usage: 30.5 MB


## Model Architecture — Masked Transformer

### Why a Transformer?

The challenge explicitly requires **amortized inference**: a single model that produces predictions in one forward pass, without fitting a new regressor per sample.

A **Transformer Encoder** is ideal here because:
- **Self-attention** allows every missing gap position to directly attend to *all* surviving context points — regardless of temporal distance
- It naturally handles variable context density (some clips may have 10 context points, others 60)
- It generalizes across instruments because it learns *how to read context*, not what a specific instrument sounds like

### Input Representation

For each of the 100 time steps, we give the model **2 features**:
1. `masked_value` — the true voltage if `Is_Context=1`, else `0.0`
2. `mask` — the `Is_Context` flag itself (`1` or `0`)

This lets the model distinguish between 'the signal is genuinely zero here' vs 'we do not know the value here.'

### Architecture Summary
```
Input [batch, 100, 2]
  -> Linear Projection  -> [batch, 100, 64]
  -> + Learnable Positional Encoding
  -> Transformer Encoder (3 layers, 4 heads, GELU activation)
  -> Linear Projection  -> [batch, 100, 1]
  -> squeeze            -> [batch, 100]
```

In [ ]:
import torch.nn as nn
class SpectralTransformer(nn.Module):
    def __init__(self, seq_len=100, feature_dim=2, d_model=64, nhead=4, num_layers=3):
        super(SpectralTransformer, self).__init__()

        # 1. Input Projection: Maps our 2 input features (Masked Value, Mask)
        # to a higher-dimensional embedding space so the Transformer can process it.
        self.input_proj = nn.Linear(feature_dim, d_model)

        # 2. Positional Encoding: Audio is sequential. Transformers don't inherently
        # know order, so we add a learnable positional embedding for the 100 time steps.
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

        # 3. Transformer Encoder: The core engine. It uses Self-Attention to let
        # the missing gaps "look" at the provided context points.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True,
            activation='gelu'  # GELU generally performs better for continuous signals
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4.Maps the high-dimensional output back down to a
        # single voltage prediction for each time step.
        self.output_proj = nn.Linear(d_model, 1)

    def forward(self, x):
        """
        x shape: [batch_size, seq_len=100, feature_dim=2]
        """
        x = self.input_proj(x)  # Shape: [batch_size, 100, d_model]
        x = x + self.pos_encoder
        x = self.transformer_encoder(x)
        out = self.output_proj(x)  # Shape becomes: [batch_size, 100, 1]
        return out.squeeze(-1)  # Final shape: [batch_size, 100]

In [ ]:
##Sanity Check
model = SpectralTransformer(seq_len=100,feature_dim=2,d_model=128,nhead=8,num_layers=4)
dummy_input = torch.randn(32, 100, 2)
dummy_output = model(dummy_input)
print(f"Output shape: {dummy_output.shape}")

Output shape: torch.Size([32, 100])


---
## Dataset Class and Loss Function

### Custom Masked MSE Loss

We only train on the **gap points** (`Is_Context = 0`), not the context points (those are already known). A standard MSE would dilute the learning signal by rewarding trivially-easy context point predictions.

The masked loss zeros out errors at context positions and averages only over the true gaps:

```
Loss = (1 / N_gaps) * sum over gaps of (y_pred - y_true)^2
```

### Dataset Design

`SpectralDataset` groups the flat CSV by `Sample_ID` and converts each clip into:
- `x`: the 2-feature input tensor of shape `[100, 2]`
- `y_true`: the full ground-truth voltage sequence of shape `[100]`
- `mask`: the `Is_Context` flags of shape `[100]`

Each clip is treated as an independent sample — no information leaks between IDs


In [ ]:
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
def masked_mse_loss(y_pred, y_true, mask):
    # mask: 1 means context (known), 0 means gap (hidden)
    # We want to calculate loss ONLY on the gaps (0), so we invert the mask.
    inverted_mask = 1.0 - mask
    squared_error = (y_pred - y_true) ** 2
    masked_error = squared_error * inverted_mask
    loss = masked_error.sum() / (inverted_mask.sum() + 1e-8)
    return loss

class SpectralDataset(Dataset):
    def __init__(self, dataframe, seq_len=100):
        self.seq_len = seq_len
        self.data_by_sample_id = []
        for sample_id, group in dataframe.groupby('Sample_ID'):
            # sorted by Time_ms
            group = group.sort_values(by='Time_ms')
            values = torch.tensor(group['Value'].values, dtype=torch.float32)
            is_context = torch.tensor(group['Is_Context'].values, dtype=torch.float32)
            self.data_by_sample_id.append({
                'y_true': values,
                'mask': is_context,
            })
    def __len__(self):
        return len(self.data_by_sample_id)
    def __getitem__(self, idx):
        sample_data = self.data_by_sample_id[idx]

        y_true_raw = sample_data['y_true']
        mask_raw = sample_data['mask']
        current_len = y_true_raw.shape[0]

        # Pad or truncate to seq_len
        if current_len < self.seq_len:
            # Pad with zeros. Adjust mask padding to be 0 as well.
            padding_len = self.seq_len - current_len
            y_true = torch.cat([y_true_raw, torch.zeros(padding_len, dtype=torch.float32)])
            mask = torch.cat([mask_raw, torch.zeros(padding_len, dtype=torch.float32)])
        elif current_len > self.seq_len:
            y_true = y_true_raw[:self.seq_len]
            mask = mask_raw[:self.seq_len]
        else:
            y_true = y_true_raw
            mask = mask_raw

        # Create input features for the transformer
        # Feature 1: Masked Value (0 if Is_Context is 0, original value if Is_Context is 1)
        # Feature 2: Is_Context (the mask itself)
        masked_value = y_true * mask
        x = torch.stack([masked_value, mask], dim=-1) # Shape: [seq_len, 2]
        return {'x': x, 'y_true': y_true, 'mask': mask}



---
## Training

### Hyperparameter Choices

| Hyperparameter | Value | Rationale |
|---|---|---|
| Model dim (`d_model`) | 64 | Lightweight but expressive for 100-step sequences |
| Attention heads | 4 | Captures multiple spectral patterns simultaneously |
| Transformer layers | 3 | Deep enough for non-stationary signals |
| Optimizer | AdamW | Better generalization than Adam for Transformers |
| Learning rate | 1e-3 | Standard starting point; decays naturally via AdamW |
| Batch size | 32 | Fits comfortably in T4 VRAM |
| Epochs | 10 | Loss converges well within this range |

### Gradient Clipping

We clip gradients at `max_norm=1.0` to prevent exploding gradients, which are common when training Transformers on continuous-valued signals.

### Preventing Memorization (Validation & Generalization)
A model that achieves zero loss on its training data runs a severe risk of overfitting—memorizing synthetic wave formulas rather than learning generalized audio physics.

To guarantee generalization for unseen offline test data in Round 2, we implemented a robust validation strategy:
1. **Validation Split:** 20% of the dataset was hidden during backpropagation to serve as an unseen benchmark.
2. **Early Stopping & Scheduling:** We utilized an `AdamW` optimizer with weight decay, coupled with a `ReduceLROnPlateau` scheduler monitoring the validation loss.

By tracking both Training and Validation MSE simultaneously, we can halt training at the exact epoch where the model achieved generalization.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
model = SpectralTransformer(seq_len=100, feature_dim=2, d_model=64, nhead=4, num_layers=3)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

dataset = SpectralDataset(df, seq_len=100)
batch_size = 32
total_dataset = len(dataset)
train_size = int(0.8 * total_dataset)
val_size = total_dataset - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 3. The Training Loop
num_epochs = 10

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    total_loss = 0.0

    # Wrap your dataloader in tqdm for a clean visual progress bar
    loop = tqdm(train_dataloader, leave=False, desc=f"Epoch {epoch+1} Train")

    for batch in loop:
        x = batch['x'].to(device)
        y_true = batch['y_true'].to(device)
        mask = batch['mask'].to(device)
        optimizer.zero_grad()

        # Forward pass
        y_pred = model(x)
        loss = masked_mse_loss(y_pred, y_true, mask)

        # Backward pass and optimize weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
    avg_train_loss = total_loss / len(train_dataloader)

    # --- VALIDATION PHASE ---
    model.eval() # Freeze weights, disable dropout
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_dataloader:
            x = batch['x'].to(device)
            y_true = batch['y_true'].to(device)
            mask = batch['mask'].to(device)
            y_pred = model(x)
            val_loss = masked_mse_loss(y_pred, y_true, mask)
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)
    scheduler.step(avg_val_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train MSE: {avg_train_loss:.6f} | Val MSE: {avg_val_loss:.6f}")
    print("-" * 60)

Training on device: cuda


Epoch [1/10] | Train MSE: 0.016027 | Val MSE: 0.000008
------------------------------------------------------------


Epoch [2/10] | Train MSE: 0.000126 | Val MSE: 0.000002
------------------------------------------------------------


Epoch [3/10] | Train MSE: 0.000074 | Val MSE: 0.000001
------------------------------------------------------------


Epoch [4/10] | Train MSE: 0.000055 | Val MSE: 0.000001
------------------------------------------------------------


...

Epoch [6/10] | Train MSE: 0.000028 | Val MSE: 0.000006
------------------------------------------------------------


Epoch [7/10] | Train MSE: 0.000024 | Val MSE: 0.000003
------------------------------------------------------------


Epoch [8/10] | Train MSE: 0.000022 | Val MSE: 0.000002
------------------------------------------------------------


Epoch [9/10] | Train MSE: 0.000020 | Val MSE: 0.000005
------------------------------------------------------------


Epoch [10/10] | Train MSE: 0.000019 | Val MSE: 0.000003
------------------------------------------------------------


In [ ]:
###Saving the submission.csv file
import numpy as np
model.eval()
inference_dataloader = DataLoader(dataset, batch_size=256, shuffle=False)
all_predictions = []
print("Running amortized inference over 80,000 samples...")
with torch.no_grad():
    for batch in inference_dataloader:
        x = batch['x'].to(device)
        y_pred = model(x)
        all_predictions.append(y_pred.cpu().numpy())
flat_predictions = np.concatenate(all_predictions, axis=0).flatten()
df['Predicted_Value'] = flat_predictions

##include the gaps (where Is_Context == 0)
submission_df = df[df['Is_Context'] == 0].copy()
submission_df = submission_df[['Sample_ID', 'Time_ms', 'Predicted_Value']]
min_val = df['Value'].min()
max_val = df['Value'].max()
submission_df['Predicted_Value'] = submission_df['Predicted_Value'].clip(lower=min_val, upper=max_val)
submission_df.to_csv('submission-50.csv', index=False)
print(f"Success! submission-4.csv created with {len(submission_df)} rows.")

Running amortized inference over 80,000 samples...
Success! submission-4.csv created with 800000 rows.


In [ ]:
torch.save(model.state_dict(), 'spect-2_model.pth')
print("Model saved!")

Model saved!
